# מפגש 21 — פתרון מורה Audited
מפתח לתלמיד על בסיס נתוני A/B/C המקוריים. אין להשתמש במחברת זו במקום מעבדת התלמיד. כל התוצאות הן דידקטיות מומצאות.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
data=pd.DataFrame({"candidate":["A","B","C"],"train":[1.00,0.88,0.60],"validation":[0.65,0.84,0.58]})
assert data.shape==(3,3)
assert list(data["candidate"])==["A","B","C"]
assert data["train"].between(0,1).all()
assert data["validation"].between(0,1).all()
assert data["candidate"].is_unique
assert data["train"].notna().all()
assert data["validation"].notna().all()

In [ ]:
data["gap"]=data["train"]-data["validation"]
assert np.allclose(data["gap"],[.35,.04,.02])
assert np.isclose(data.loc[0,"gap"],.35)
assert np.isclose(data.loc[1,"gap"],.04)
assert np.isclose(data.loc[2,"gap"],.02)
assert data["gap"].notna().all()
assert len(data["gap"].unique())==3
print(data)

## בחירה — נתוני Validation ולא Train
לפי תנאי המקור המדד גדול יותר כשהמודל מצליח יותר. בוחרים את המועמד שמוביל באימות; ההובלה באימון לבדה אינה הקריטריון.

In [ ]:
candidate=data.loc[data["validation"].idxmax(),"candidate"]
train_candidate=data.loc[data["train"].idxmax(),"candidate"]
assert candidate=="B"
assert train_candidate=="A"
assert candidate!=train_candidate
assert np.isclose(data["validation"].max(),.84)
assert np.isclose(data.loc[data["candidate"]=="B","train"].iloc[0],.88)
assert np.isclose(data.loc[data["candidate"]=="C","validation"].iloc[0],.58)
assert data.loc[data["candidate"]==candidate,"gap"].iloc[0]>0
print("Chosen:",candidate)

## פרשנות שמבדילה ראיה מטענה
A מציג פער גדול ולכן חשד אפשרי להתאמת יתר. C מציג ביצועים נמוכים באימון ובאימות ולכן חשד אפשרי להתאמת חסר. **לא** מוכיחים סיבה ללא נתונים נוספים, וגם לא מבטיחים הכללה בגלל פער קטן.

In [ ]:
assert data.loc[0,"gap"]>data.loc[1,"gap"]
assert data.loc[2,"train"]<data.loc[1,"train"]
assert data.loc[2,"validation"]<data.loc[1,"validation"]
assert data.loc[2,"gap"]<data.loc[0,"gap"]
assert data["train"].between(0,1).all()
assert data["validation"].between(0,1).all()

## תפקיד Test ותזמון
נעלו החלטה לפני חשיפת Test. החלפת seed לאחר צפייה ב־Test לא הופכת את ההחלטה לבלתי תלויה.

In [ ]:
roles={"Train":"fit","Validation":"choose","Test":"final"}
steps=["metric fixed","validation comparison","choice locked","test evaluation"]
assert len(set(roles.values()))==3
assert steps.index("choice locked")<steps.index("test evaluation")
assert steps.index("validation comparison")<steps.index("choice locked")
assert "test selection" not in steps
assert len(steps)==4

## בדיקת עזר לחלוקה סכמטית בלבד
הנתונים הבאים הם מזהים ולא ציוני המועמדים. הם ממחישים שאין רשומה משותפת ל־Train ול־Test.

In [ ]:
from sklearn.model_selection import train_test_split
ids=np.arange(20)
train_ids,test_ids=train_test_split(ids,test_size=.25,random_state=42)
assert len(train_ids)==15
assert len(test_ids)==5
assert not (set(train_ids)&set(test_ids))
assert len(set(train_ids)|set(test_ids))==20
assert sorted(list(train_ids)+list(test_ids))==list(ids)
print("Split checks passed")

## פתרון מילולי למורה
מועמד בעל Train=0.48 ו־Validation=0.47 אינו מצטיין רק בגלל פער 0.01. אין ציון Test, ולכן אין להמציא אחד. בכרטיס ההגנה התלמיד מסביר מתי נקבעת הבחירה, איך מונעים דליפה ולמה ההחלטה צריכה להיות מתועדת.